# Lot D — S4 : recherche dense multilingue (EXP-D-003)

**Auteur : Membre 5.** Compare BM25, la recherche dense et l'hybride (RRF) sur le jeu de dev
`tests/fixtures/rag_llm/`, pour plusieurs modèles d'embeddings.

1. *Exécution → Modifier le type d'exécution → GPU T4* (conseillé pour `bge-m3`, facultatif sinon).
2. Exécuter les cellules dans l'ordre.
3. Recopier les tableaux « Comparaison (GLOBAL) » et les seuils dans `experiments.md` (EXP-D-003).

## 1. Récupérer le projet
Dépôt privé : remplacer `<TOKEN>` par un *Personal access token* GitHub (lecture), ou téléverser le zip du projet et le décompresser.

In [ ]:
!git clone -b lot-d-rag https://github.com/SouissiOumaima/NewsCheckTN.git
# Dépôt privé :
# !git clone -b lot-d-rag https://<TOKEN>@github.com/SouissiOumaima/NewsCheckTN.git
%cd NewsCheckTN

In [ ]:
!pip install -q -r requirements-dev.txt sentence-transformers

## 2. BM25 vs dense vs hybride — modèle par défaut (MiniLM multilingue)

In [ ]:
!python -m rag_llm.evaluate_retrieval --method all

## 3. Autres modèles d'embeddings
`e5` ajoute automatiquement les préfixes `query:` / `passage:`. `bge-m3` (≈ 2 Go) demande le GPU.

In [ ]:
for model in ["intfloat/multilingual-e5-small",
              "intfloat/multilingual-e5-base",
              "BAAI/bge-m3"]:
    print("\n" + "#" * 80 + f"\n# {model}")
    !python -m rag_llm.evaluate_retrieval --method all --model {model}

## 4. Calibrer le seuil d'abstention dense

Pour chaque modèle, la sortie « dense » affiche :
- *Score du 1er passage quand il est correct* (min) ;
- *Affirmations sans preuve dans le corpus* (`dev-c16`, `dev-c17`).

Le seuil `MIN_DENSE_SIMILARITY` doit être **entre** le score des affirmations sans preuve et le min
des bonnes preuves. On le teste ci-dessous (variable d'environnement, sans modifier le code).

In [ ]:
import os
for seuil in ["0.3", "0.4", "0.5", "0.6"]:
    os.environ["MIN_DENSE_SIMILARITY"] = seuil
    print(f"\n===== MIN_DENSE_SIMILARITY = {seuil}")
    !MIN_DENSE_SIMILARITY={seuil} python -m rag_llm.evaluate_verdict --method hybrid | tail -10

## 5. Verdict de bout en bout avec le meilleur retrieval
Remplacer le modèle par celui qui a le meilleur Recall@5 à l'étape 3.

In [ ]:
!python -m rag_llm.evaluate_verdict --method bm25 | tail -9
!python -m rag_llm.evaluate_verdict --method hybrid --model intfloat/multilingual-e5-base | tail -9

## 6. Résultats
Les fichiers JSON sont dans `evaluation/results/` (non versionnés) : les télécharger depuis le panneau Fichiers si besoin.

In [ ]:
!ls evaluation/results/